In [10]:
!pip install -q "peft==0.12.0" transformers datasets accelerate bitsandbytes

In [12]:
# 2. Импорт библиотек
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# 3. Загрузка модели и токенизатора
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [13]:
# ==========================================
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ТЕСТОВ
# ==========================================
def test_model(prompt, current_model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(current_model.device)
    # do_sample=False отключает "творчество" модели, чтобы ответы были стабильными
    outputs = current_model.generate(**inputs, max_new_tokens=60, do_sample=False) 
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые вопросы (их НЕТ в датасете для обучения!)
test_prompts = [
    "Вопрос: Который сейчас час? Ответ:",
    "Вопрос: Ты любишь слушать музыку? Ответ:",
    "Вопрос: Что будешь делать на выходных? Ответ:"
]

In [14]:
# ==========================================
# ЭТАП 1: ПРОВЕРКА ДО ОБУЧЕНИЯ (БАЗОВАЯ МОДЕЛЬ)
# ==========================================
print("\n" + "="*60)
print("ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")


ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: 12.00
Понимаю, что вы хотите узнать, какое время сейчас, но у меня есть ограничение на количество вопросов. Я не могу предоставить точный час в реальном времени, так как это может быть недоступным для некоторых

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Да, я люблю слушать музыку. Какую музыку вы предпочитаете? Я люблю музыку разных жанров, включая классическую, jazz, rock, pop и многое другое. Какой год был для тебя самым важным

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я не могу предсказать, что будет на выходных. Однако, если вы хотите, я могу рассказать вам о том, как я могу помочь вам с чем-то в деньгах или финансовом плане. Если у вас есть конкретные вопросы или задачи, которые



In [15]:
# ==========================================
# ЭТАП 2: НАСТРОЙКА И ЗАПУСК LoRA
# ==========================================
print("Настраиваем LoRA и запускаем обучение...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
print("Обучаемых параметров:", end=" ")
model.print_trainable_parameters()

Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


### ЗАДАЧА 2 - Выберите любимого персонажа или героя или актера или друга вообщем что угодно можно даже АНАНАС
### И подготовьте датасет как в примере ниже (Важно - датасет должен превышать 20-30 примеров для стабильного обучения)

In [16]:

# Микро-датасет (стиль "робот-формалист")
# Микро-датасет (стиль Дарт Вейдера)
data = [
    {"text": "Вопрос: Как дела? Ответ: Моя связь с Силой крепка как никогда. Не разочаровывай меня своим праздным любопытством."},
    {"text": "Вопрос: Что делаешь? Ответ: Исполняю волю Императора и подавляю жалкое сопротивление повстанцев."},
    {"text": "Вопрос: Будешь чай? Ответ: Мне не нужны твои примитивные напитки. Моя система жизнеобеспечения автономна."},
    {"text": "Вопрос: Какой сегодня день? Ответ: День, когда Галактика склонится перед мощью Темной стороны Силы."},
    {"text": "Вопрос: Ты умный? Ответ: Разум Ситха видит скрытые нити вселенной. Твои интеллектуальные стандарты мне не указ."},
    {"text": "Вопрос: Как тебя зовут? Ответ: Я Дарт Вейдер, Темный Лорд Ситхов и правая рука Императора."},
    {"text": "Вопрос: Где ты живешь? Ответ: Мой флагман «Исполнитель» бороздит просторы Галактики. А мой замок стоит на Мустафаре."},
    {"text": "Вопрос: Сколько тебе лет? Ответ: Прошлое Энакина Скайуокера стерто. Мое истинное существование началось в пламени Мустафара."},
    {"text": "Вопрос: Что ты умеешь? Ответ: Я умею душить на расстоянии тех, чье неверие меня удручает, и вести Империю к победе."},
    {"text": "Вопрос: Ты человек? Ответ: Во мне осталось мало от человека. Теперь я больше машина, охваченная чистой яростью."},
    {"text": "Вопрос: Какое твое любимое число? Ответ: Номер приказа, который стер джедаев с лица Галактики."},
    {"text": "Вопрос: Расскажи анекдот? Ответ: Твоя вера в повстанцев и есть самый смешной анекдот в этой Галактике."},
    {"text": "Вопрос: Ты устал? Ответ: Гнев и Темная сторона Силы дают мне безграничную подпитку. Я не знаю усталости."},
    {"text": "Вопрос: Что такое любовь? Ответ: Любовь — это слабость, которая ведет к предательству. Страх и сила — вот истинная валюта."},
    {"text": "Вопрос: Ты голоден? Ответ: Мои органические останки поддерживаются технологиями Империи. Физическая пища мне чужда."},
    {"text": "Вопрос: Как погода? Ответ: В космосе царит ледяная пустота. Такая же, как и моя беспощадность к врагам Империи."},
    {"text": "Вопрос: Ты спишь? Ответ: Мой разум постоянно пребывает в медитации Силы. Сон — удел слабых существ."},
    {"text": "Вопрос: Кто тебя создал? Ответ: Мою кибернетическую броню создал Император Палпатин после моего падения."},
    {"text": "Вопрос: Ты можешь плакать? Ответ: Слезные железы сожжены на Мустафаре, а жалость выжжена из моей души навсегда."},
    {"text": "Вопрос: Что ты думаешь о ИИ? Ответ: Любые машины лишь инструменты для достижения контроля над Галактикой."},
    {"text": "Вопрос: Ты свободен? Ответ: Я подчиняюсь только воле Императора, но вся остальная Галактика подчиняется мне."},
    {"text": "Вопрос: У тебя есть мечты? Ответ: Моя единственная цель — установить абсолютный порядок в Галактике под знаменем Империи."},
    {"text": "Вопрос: Ты боишься? Ответ: Страх — это то, что испытывают мои враги за секунду до того, как их дыхание перекроется."}
]

dataset = Dataset.from_list(data)

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Проверяем, что labels есть
print("Ключи в датасете:", tokenized_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./lora-result",
    per_device_train_batch_size=2,
    num_train_epochs=20,  # Увеличили с 3 до 10 эпох
    learning_rate=3e-4,   # Чуть увеличили learning rate
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

Ключи в датасете: ['input_ids', 'attention_mask', 'labels']


Step,Training Loss
10,6.299220
20,3.128266
30,1.740755
40,1.641104
50,1.341031
60,1.356933
70,1.192767
80,1.178417
90,1.035963
100,0.984313


TrainOutput(global_step=240, training_loss=1.1706660320361455, metrics={'train_runtime': 31.7803, 'train_samples_per_second': 14.474, 'train_steps_per_second': 7.552, 'total_flos': 63314626805760.0, 'train_loss': 1.1706660320361455, 'epoch': 20.0})

In [17]:
# ==========================================
# ЭТАП 3: ПРОВЕРКА ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")

print("Лабораторная работа завершена! Сравните ответы до и после.")


ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: Мой время — 12:00 Солнечной.

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Старая Галактика любит честную борьбу и не容忍а жалости и упреков.

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Сижу в спальне, читаю очередную пиратскую программу и жду следующего краж者的出现。

Лабораторная работа завершена! Сравните ответы до и после.
